In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_json_path = 'rca100/cases/t008/task.json'
with open(task_json_path, 'r') as file:
    task_data = json.load(file)

# Extract relevant information from the task.json
alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is titled "cart响应时间抖动告警" (Cart Response Time Jitter Alert). It occurred between 2026-04-13 04:22:01 and 2026-04-13 04:28:03 (UTC+8). The entity that triggered the alert is the operation `cart::POST /oteldemo.CartService/AddItem`, which belongs to the APM domain. The user is requesting help to analyze the root cause of this issue.

The original code execution output of IPython Kernel is also provided below for reference:

('cart响应时间抖动告警', {'start': '2026-04-13T04:22:01+08:00', 'end': '2026-04-13T04:28:03+08:00'}, {'entity_id': 'e433b3f842dc4e525a834701cc603c28', 'entity_name': 'cart::POST /oteldemo.CartService/AddItem', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">cart响应时间抖动告警<alert_event event_id="e7ea97981cf940f6180f3de7d392a069" trans_id="c0hjo0m6c93u30cugghgfjqb01" rule_id="40ef4477-f6e2-414e-8307-1009a8e45c35" rule_name="cart响应时间抖动告警" /></code>\n\n帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_path = 'rca100/cases/t008/metrics.parquet'
metrics_df = pd.read_parquet(metrics_path)

# List all available KPIs and their corresponding entity sets
kpi_entity_sets = metrics_df[['metric', 'entity_set']].drop_duplicates()

kpi_entity_sets

```
Out[2]:
```
The `metrics.parquet` file contains 63 unique KPIs across various entity sets. Relevant KPIs for anomaly detection in the cart service and other components include metrics like `error_rate`, `latency`, `request_count`, `error_count`, `slow`, and `workload`. These KPIs are associated with entity sets such as `apm.service.legacy`, `apm.operation`, and `apm.instance`. For node-level analysis, metrics like `node_memory_usage_rate`, `node_disk_usage_rate`, and `node_pod_running_count` under the `k8s.node` entity set are also available.

The original code execution output of IPython Kernel is also provided below for reference:

metric          entity_set
0           node_ready_status            k8s.node
2055    node_not_ready_status            k8s.node
2192   node_memory_usage_rate            k8s.node
6028     node_disk_usage_rate            k8s.node
6165   node_pod_running_count            k8s.node
...                       ...                 ...
75108                workload  apm.service.legacy
75120                    slow  apm.service.legacy
75135                   error  apm.service.legacy
84695              error_rate       apm.operation
84697              error_rate        apm.instance

[63 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

# Ensure 'time' and 'value' columns are numeric for calculations
metrics_df['time'] = pd.to_numeric(metrics_df['time'], errors='coerce')
metrics_df['value'] = pd.to_numeric(metrics_df['value'], errors='coerce')

# Create a new column to classify by 'component-KPI'
metrics_df['component_kpi'] = metrics_df['entity_name'] + '-' + metrics_df['metric']

# Group by 'component-KPI' and calculate the global P95 threshold for each time series
p95_thresholds = metrics_df.groupby('component_kpi')['value'].quantile(0.95).reset_index()
p95_thresholds.columns = ['component_kpi', 'p95_threshold']

p95_thresholds

```
Out[3]:
```
The global P95 thresholds have been calculated for 272 unique 'component-KPI' time series in the `metrics.parquet` file. These thresholds represent the 95th percentile of metric values for each component and KPI combination. This data can be used to identify anomalies by comparing actual metric values against these thresholds, particularly for the cart service and other potential root cause components.

The original code execution output of IPython Kernel is also provided below for reference:

component_kpi  p95_threshold
0                    -apiserver_concurrent_connections       6.000000
1                  -apiserver_non_2xx_read_request_qps       0.267261
2                 -apiserver_non_2xx_write_request_qps       0.053723
3                       -apiserver_requests_per_second      21.246175
4                           -cluster_failed_node_count       0.000000
..                                                 ...            ...
267                                  shipping-workload    4668.600000
268  shipping::oteldemo.ShippingService/GetQuote-av...       0.002863
269  shipping::oteldemo.ShippingService/GetQuote-re...    2334.000000
270  shipping::oteldemo.ShippingService/ShipOrder-a...       0.000018
271  shipping::oteldemo.ShippingService/ShipOrder-r...    2334.600000

[272 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pytz
from datetime import datetime

# Define the failure duration in UTC+8
start_time = datetime.fromisoformat('2026-04-13T04:22:01+08:00').astimezone(pytz.timezone('Asia/Shanghai'))
end_time = datetime.fromisoformat('2026-04-13T04:28:03+08:00').astimezone(pytz.timezone('Asia/Shanghai'))

# Convert the failure duration to microseconds for filtering
start_time_us = int(start_time.timestamp() * 1_000_000)
end_time_us = int(end_time.timestamp() * 1_000_000)

# Filter metrics data within the failure duration
filtered_metrics = metrics_df[(metrics_df['time'] >= start_time_us) & (metrics_df['time'] <= end_time_us)]

# Merge the filtered metrics with the P95 thresholds
anomalies = filtered_metrics.merge(p95_thresholds, on='component_kpi', how='left')

# Identify anomalies where the metric value exceeds the P95 threshold
anomalies = anomalies[anomalies['value'] > anomalies['p95_threshold']]

anomalies

```
Out[4]:
```
During the failure duration (2026-04-13 04:22:01 to 2026-04-13 04:28:03 UTC+8), 295 anomalies were identified where metric values exceeded their global P95 thresholds. These anomalies span multiple components and KPIs, including node-level metrics like `node_disk_usage_rate` and service-level metrics like `latency` and `mem` for services such as `email`, `frontend-proxy`, `frontend`, `image-provider`, and `inventory`. These anomalies indicate potential performance issues in these components during the incident.

The original code execution output of IPython Kernel is also provided below for reference:

time domain          entity_set entity_id     entity_name  ...         value                                     metric_set_id service           component_kpi p95_threshold
650   1776025321000000    k8s            k8s.node                            ...  3.627293e+01  k8s@metric_set@k8s.metric.high_level_metric_node    None   -node_disk_usage_rate  3.625248e+01
651   1776025351000000    k8s            k8s.node                            ...  3.627981e+01  k8s@metric_set@k8s.metric.high_level_metric_node    None   -node_disk_usage_rate  3.625248e+01
652   1776025381000000    k8s            k8s.node                            ...  3.627995e+01  k8s@metric_set@k8s.metric.high_level_metric_node    None   -node_disk_usage_rate  3.625248e+01
653   1776025411000000    k8s            k8s.node                            ...  3.628040e+01  k8s@metric_set@k8s.metric.high_level_metric_node    None   -node_disk_usage_rate  3.625248e+01
654   1776025441000000    k8s            k8s.node                            ...  3.628068e+01  k8s@metric_set@k8s.metric.high_level_metric_node    None   -node_disk_usage_rate  3.625248e+01
...                ...    ...                 ...       ...             ...  ...           ...                                               ...     ...                     ...           ...
8203  1776025681000000    apm  apm.service.legacy                     email  ...  6.144328e+08                                              None    None               email-mem  5.914867e+08
8211  1776025681000000    apm  apm.service.legacy            frontend-proxy  ...  3.600372e-02                                              None    None  frontend-proxy-latency  3.188761e-02
8216  1776025681000000    apm  apm.service.legacy                  frontend  ...  3.487025e-02                                              None    None        frontend-latency  3.095042e-02
8220  1776025681000000    apm  apm.service.legacy            image-provider  ...  1.131422e-04                                              None    None      image-provider-cpu  1.113966e-04
8224  1776025681000000    apm  apm.service.legacy                 inventory  ...  1.351032e-03                                              None    None       inventory-latency  1.342895e-03

[295 rows x 11 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_path = 'rca100/cases/t008/traces.parquet'
traces_df = pd.read_parquet(traces_path)

# Ensure relevant columns are numeric for filtering
traces_df['startTime'] = pd.to_numeric(traces_df['startTime'], errors='coerce')
traces_df['endTime'] = pd.to_numeric(traces_df['endTime'], errors='coerce')

# Filter traces within the failure duration
filtered_traces = traces_df[
    (traces_df['startTime'] >= start_time_us * 1_000) & 
    (traces_df['endTime'] <= end_time_us * 1_000)
]

# Identify traces related to the 'cart::POST /oteldemo.CartService/AddItem' operation
cart_traces = filtered_traces[filtered_traces['spanName'] == 'cart::POST /oteldemo.CartService/AddItem']

# Analyze the call chain to determine the most downstream faulty service
# A downstream service is identified by its `parentSpanId` matching the `spanId` of an upstream service
downstream_faults = cart_traces.merge(
    filtered_traces, 
    left_on='spanId', 
    right_on='parentSpanId', 
    suffixes=('_upstream', '_downstream')
)

downstream_faults

```
Out[5]:
```
No traces related to the `cart::POST /oteldemo.CartService/AddItem` operation were found within the failure duration (2026-04-13 04:22:01 to 2026-04-13 04:28:03 UTC+8). As a result, the call chain could not be analyzed, and no downstream faulty service could be identified from the trace data.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [traceId_upstream, spanId_upstream, parentSpanId_upstream, kind_upstream, spanName_upstream, startTime_upstream, endTime_upstream, duration_upstream, serviceName_upstream, pid_upstream, ip_upstream, hostname_upstream, statusCode_upstream, statusMessage_upstream, traceState_upstream, resources_upstream, attributes_upstream, __topic___upstream, __tag__:__pack_id___upstream, events_upstream, links_upstream, traceId_downstream, spanId_downstream, parentSpanId_downstream, kind_downstream, spanName_downstream, startTime_downstream, endTime_downstream, duration_downstream, serviceName_downstream, pid_downstream, ip_downstream, hostname_downstream, statusCode_downstream, statusMessage_downstream, traceState_downstream, resources_downstream, attributes_downstream, __topic___downstream, __tag__:__pack_id___downstream, events_downstream, links_downstream]
Index: []

[0 rows x 42 columns]```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_path = 'rca100/cases/t008/logs.parquet'
logs_df = pd.read_parquet(logs_path)

# Ensure '_time_' column is numeric for filtering
logs_df['_time_'] = pd.to_numeric(logs_df['_time_'], errors='coerce')

# Filter logs within the failure duration
filtered_logs = logs_df[(logs_df['_time_'] >= start_time_us) & (logs_df['_time_'] <= end_time_us)]

# Search for logs related to the 'cart' service and look for error messages or anomalies
cart_logs = filtered_logs[filtered_logs['_container_name_'].str.contains('cart', case=False, na=False)]
error_logs = cart_logs[cart_logs['content'].str.contains('error', case=False, na=False)]

error_logs

```
Out[6]:
```
No error logs related to the `cart` service were found within the failure duration (2026-04-13 04:22:01 to 2026-04-13 04:28:03 UTC+8). This indicates that no explicit error messages or anomalies were logged by the `cart` service during the incident.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _image_name_, _container_name_, _pod_name_, _namespace_, _pod_uid_, _container_ip_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```